In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


SEPARAR DATASET EN ACTIVOS / ELIMINADOS

In [2]:
# Importando la biblioteca pandas para manipulación y análisis de datos
import pandas as pd
business_payments = pd.read_csv('drive/MyDrive/ColabNotebooks/Business_Payments/merged_outer.csv')

In [ ]:
business_payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21057 entries, 0 to 21056
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id                          21057 non-null  int64  
 1   cash_request_id             21057 non-null  int64  
 2   type                        21057 non-null  object 
 3   status                      21057 non-null  object 
 4   category                    2196 non-null   object 
 5   total_amount                21057 non-null  float64
 6   reason                      21057 non-null  object 
 7   created_at                  21057 non-null  object 
 8   updated_at                  21057 non-null  object 
 9   paid_at                     15438 non-null  object 
 10  from_date                   6749 non-null   object 
 11  to_date                     6512 non-null   object 
 12  charge_moment               21057 non-null  object 
 13  amount                      210

In [ ]:
import pandas as pd

# Cargar el archivo CSV original
file_path = 'drive/MyDrive/ColabNotebooks/Business_Payments/merged_inner.csv'  # Cambia esta ruta según corresponda
data = pd.read_csv(file_path)

# Separar los datos en usuarios activos y eliminados
usuarios_activos = data[data['user_id'].notna()]
usuarios_eliminados = data[data['deleted_account_id'].notna()]

# Guardar los datos en archivos CSV separados
usuarios_activos.to_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_activos.csv', index=False)
usuarios_eliminados.to_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_eliminados.csv.csv', index=False)

print("Archivos generados: 'usuarios_activos.csv' y 'usuarios_eliminados.csv'")


Archivos generados: 'usuarios_activos.csv' y 'usuarios_eliminados.csv'


--------------------------------------------------------------------------------------------------------

INGENIERIA DE *DATOS*



In [3]:
import pandas as pd

In [30]:
usuarios_activos = pd.read_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_activos.csv')
usuarios_eliminados = pd.read_csv('drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_eliminados.csv')

In [31]:
# Eliminar columnas no necesarias
usuarios_activos = usuarios_activos.drop(columns=['deleted_account_id'], errors='ignore')
usuarios_eliminados = usuarios_eliminados.drop(columns=['user_id'], errors='ignore')

In [7]:
usuarios_activos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29522 entries, 0 to 29521
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id_fees                     20151 non-null  float64
 1   cash_request_id             20151 non-null  float64
 2   type                        20151 non-null  object 
 3   status_fees                 20151 non-null  object 
 4   category                    2030 non-null   object 
 5   total_amount                20151 non-null  float64
 6   reason                      20151 non-null  object 
 7   created_at_fees             20151 non-null  object 
 8   updated_at_fees             20151 non-null  object 
 9   paid_at                     15016 non-null  object 
 10  from_date                   7291 non-null   object 
 11  to_date                     7291 non-null   object 
 12  charge_moment               20151 non-null  object 
 13  id_cash_request             295

In [32]:
import pandas as pd

# Seleccionar la columna correcta para la conversión de fecha
fecha_column_activos = "created_at_cash_request"
fecha_column_eliminados = "created_at_cash_request"

# Convertir a datetime si aún no lo es
usuarios_activos[fecha_column_activos] = pd.to_datetime(usuarios_activos[fecha_column_activos], errors="coerce")
usuarios_eliminados[fecha_column_eliminados] = pd.to_datetime(usuarios_eliminados[fecha_column_eliminados], errors="coerce")

# Verificar si ya tienen zona horaria asignada y asignar si es necesario
timezone_origen = "Europe/Madrid"  # Cambia esto si es otro timezone

if usuarios_activos[fecha_column_activos].dt.tz is None:
    usuarios_activos[fecha_column_activos] = usuarios_activos[fecha_column_activos].dt.tz_localize(timezone_origen)

if usuarios_eliminados[fecha_column_eliminados].dt.tz is None:
    usuarios_eliminados[fecha_column_eliminados] = usuarios_eliminados[fecha_column_eliminados].dt.tz_localize(timezone_origen)

# Convertir a UTC
usuarios_activos[fecha_column_activos] = usuarios_activos[fecha_column_activos].dt.tz_convert("UTC")
usuarios_eliminados[fecha_column_eliminados] = usuarios_eliminados[fecha_column_eliminados].dt.tz_convert("UTC")

# Verificar resultado
print(usuarios_activos[[fecha_column_activos]].head())
print(usuarios_eliminados[[fecha_column_eliminados]].head())


           created_at_cash_request
0 2019-11-19 13:57:53.511561+00:00
1 2019-12-10 19:05:21.596873+00:00
2 2019-12-10 19:05:48.921042+00:00
3 2019-12-10 19:13:35.825460+00:00
4 2019-12-10 19:14:41.668754+00:00
           created_at_cash_request
0 2019-12-09 14:47:35.190714+00:00
1 2019-12-10 22:23:12.452603+00:00
2 2019-12-11 07:30:42.567035+00:00
3 2019-12-11 11:40:44.978306+00:00
4 2019-12-11 14:01:02.506996+00:00


COLUMNA TOTAL SOLICITUDES

In [33]:
import pandas as pd

# Contar solicitudes por usuario en usuarios activos
solicitudes_activos = usuarios_activos["user_id"].value_counts().reset_index()
solicitudes_activos.columns = ["user_id", "total_solicitudes_usuario"]

# Unir la información al dataset original
usuarios_activos = usuarios_activos.merge(solicitudes_activos, on="user_id", how="left")

# Contar solicitudes por usuario en usuarios eliminados
solicitudes_eliminados = usuarios_eliminados["deleted_account_id"].value_counts().reset_index()
solicitudes_eliminados.columns = ["deleted_account_id", "total_solicitudes_usuario"]

# Unir la información al dataset original
usuarios_eliminados = usuarios_eliminados.merge(solicitudes_eliminados, on="deleted_account_id", how="left")



COLUMNA TOTAL OPERACIONES CANCELADAS O RECHAZADAS

In [34]:
import pandas as pd

# Contar operaciones canceladas o rechazadas en usuarios activos
canceladas_rechazadas_activos = usuarios_activos[
    usuarios_activos["status_cash_request"].isin(["cancelled", "rejected"])
]["user_id"].value_counts().reset_index()
canceladas_rechazadas_activos.columns = ["user_id", "total_operaciones_canceladas_rechazadas"]

# Unir la información al dataset original
usuarios_activos = usuarios_activos.merge(canceladas_rechazadas_activos, on="user_id", how="left")
usuarios_activos["total_operaciones_canceladas_rechazadas"].fillna(0, inplace=True)

# Contar operaciones canceladas o rechazadas en usuarios eliminados
canceladas_rechazadas_eliminados = usuarios_eliminados[
    usuarios_eliminados["status_cash_request"].isin(["cancelled", "rejected"])
]["deleted_account_id"].value_counts().reset_index()
canceladas_rechazadas_eliminados.columns = ["deleted_account_id", "total_operaciones_canceladas_rechazadas"]

# Unir la información al dataset original
usuarios_eliminados = usuarios_eliminados.merge(canceladas_rechazadas_eliminados, on="deleted_account_id", how="left")
usuarios_eliminados["total_operaciones_canceladas_rechazadas"].fillna(0, inplace=True)

# Verificar resultado
print(usuarios_activos[["user_id", "total_operaciones_canceladas_rechazadas"]].head())
print(usuarios_eliminados[["deleted_account_id", "total_operaciones_canceladas_rechazadas"]].head())


   user_id  total_operaciones_canceladas_rechazadas
0     47.0                                      1.0
1    804.0                                      1.0
2    812.0                                      3.0
3    191.0                                      1.0
4    430.0                                      0.0
   deleted_account_id  total_operaciones_canceladas_rechazadas
0              1309.0                                      0.0
1              4217.0                                      0.0
2                91.0                                      1.0
3               972.0                                      1.0
4              3324.0                                      3.0


<ipython-input-34-f467ae36c206>:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  usuarios_activos["total_operaciones_canceladas_rechazadas"].fillna(0, inplace=True)
<ipython-input-34-f467ae36c206>:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].m

COLUMNA MES DE LA SOLICITUD

In [35]:
import pandas as pd

# Seleccionar la columna correcta de fecha
fecha_column_activos = "created_at_cash_request"
fecha_column_eliminados = "created_at_cash_request"

# Convertir a zona horaria UTC
usuarios_activos[fecha_column_activos] = pd.to_datetime(usuarios_activos[fecha_column_activos], utc=True, errors="coerce")
usuarios_eliminados[fecha_column_eliminados] = pd.to_datetime(usuarios_eliminados[fecha_column_eliminados], utc=True, errors="coerce")

# Extraer el mes en formato YYYY-mm
usuarios_activos["mes_solicitud"] = usuarios_activos[fecha_column_activos].dt.strftime("%Y-%m")
usuarios_eliminados["mes_solicitud"] = usuarios_eliminados[fecha_column_eliminados].dt.strftime("%Y-%m")

# Verificar resultado
print(usuarios_activos[["mes_solicitud"]].head())
print(usuarios_eliminados[["mes_solicitud"]].head())


  mes_solicitud
0       2019-11
1       2019-12
2       2019-12
3       2019-12
4       2019-12
  mes_solicitud
0       2019-12
1       2019-12
2       2019-12
3       2019-12
4       2019-12


In [36]:
import pandas as pd

# Seleccionar la columna correcta de fecha
fecha_column_activos = "created_at_cash_request"
fecha_column_eliminados = "created_at_cash_request"

# Convertir a zona horaria UTC
usuarios_activos[fecha_column_activos] = pd.to_datetime(usuarios_activos[fecha_column_activos], utc=True, errors="coerce")
usuarios_eliminados[fecha_column_eliminados] = pd.to_datetime(usuarios_eliminados[fecha_column_eliminados], utc=True, errors="coerce")

# Extraer la semana, el nombre del mes y el año
usuarios_activos["semana_solicitud"] = usuarios_activos[fecha_column_activos].dt.isocalendar().week.astype(str) + "_" + \
                                       usuarios_activos[fecha_column_activos].dt.strftime("%B") + "_" + \
                                       usuarios_activos[fecha_column_activos].dt.strftime("%Y")

usuarios_eliminados["semana_solicitud"] = usuarios_eliminados[fecha_column_eliminados].dt.isocalendar().week.astype(str) + "_" + \
                                          usuarios_eliminados[fecha_column_eliminados].dt.strftime("%B") + "_" + \
                                          usuarios_eliminados[fecha_column_eliminados].dt.strftime("%Y")

# Verificar resultado
print(usuarios_activos[["semana_solicitud"]].head())
print(usuarios_eliminados[["semana_solicitud"]].head())



   semana_solicitud
0  47_November_2019
1  50_December_2019
2  50_December_2019
3  50_December_2019
4  50_December_2019
   semana_solicitud
0  50_December_2019
1  50_December_2019
2  50_December_2019
3  50_December_2019
4  50_December_2019


In [37]:
import pandas as pd

# Seleccionar la columna correcta de fecha
fecha_column_activos = "created_at_cash_request"
fecha_column_eliminados = "created_at_cash_request"

# Convertir a zona horaria UTC
usuarios_activos[fecha_column_activos] = pd.to_datetime(usuarios_activos[fecha_column_activos], utc=True, errors="coerce")
usuarios_eliminados[fecha_column_eliminados] = pd.to_datetime(usuarios_eliminados[fecha_column_eliminados], utc=True, errors="coerce")

# Extraer el día de la semana, la semana del año, el nombre del mes y el año
usuarios_activos["dia_semana_solicitud"] = usuarios_activos[fecha_column_activos].dt.strftime("%A") + "_" + \
                                           usuarios_activos[fecha_column_activos].dt.isocalendar().week.astype(str) + "_" + \
                                           usuarios_activos[fecha_column_activos].dt.strftime("%B") + "_" + \
                                           usuarios_activos[fecha_column_activos].dt.strftime("%Y")

usuarios_eliminados["dia_semana_solicitud"] = usuarios_eliminados[fecha_column_eliminados].dt.strftime("%A") + "_" + \
                                              usuarios_eliminados[fecha_column_eliminados].dt.isocalendar().week.astype(str) + "_" + \
                                              usuarios_eliminados[fecha_column_eliminados].dt.strftime("%B") + "_" + \
                                              usuarios_eliminados[fecha_column_eliminados].dt.strftime("%Y")

# Verificar resultado
print(usuarios_activos[["user_id", fecha_column_activos, "dia_semana_solicitud"]].head())
print(usuarios_eliminados[["deleted_account_id", fecha_column_eliminados, "dia_semana_solicitud"]].head())



   user_id          created_at_cash_request      dia_semana_solicitud
0     47.0 2019-11-19 13:57:53.511561+00:00  Tuesday_47_November_2019
1    804.0 2019-12-10 19:05:21.596873+00:00  Tuesday_50_December_2019
2    812.0 2019-12-10 19:05:48.921042+00:00  Tuesday_50_December_2019
3    191.0 2019-12-10 19:13:35.825460+00:00  Tuesday_50_December_2019
4    430.0 2019-12-10 19:14:41.668754+00:00  Tuesday_50_December_2019
   deleted_account_id          created_at_cash_request  \
0              1309.0 2019-12-09 14:47:35.190714+00:00   
1              4217.0 2019-12-10 22:23:12.452603+00:00   
2                91.0 2019-12-11 07:30:42.567035+00:00   
3               972.0 2019-12-11 11:40:44.978306+00:00   
4              3324.0 2019-12-11 14:01:02.506996+00:00   

         dia_semana_solicitud  
0     Monday_50_December_2019  
1    Tuesday_50_December_2019  
2  Wednesday_50_December_2019  
3  Wednesday_50_December_2019  
4  Wednesday_50_December_2019  


In [38]:
import pandas as pd

# Seleccionar la columna correcta de fecha
fecha_column_activos = "created_at_cash_request"
fecha_column_eliminados = "created_at_cash_request"

# Convertir a zona horaria UTC
usuarios_activos[fecha_column_activos] = pd.to_datetime(usuarios_activos[fecha_column_activos], utc=True, errors="coerce")
usuarios_eliminados[fecha_column_eliminados] = pd.to_datetime(usuarios_eliminados[fecha_column_eliminados], utc=True, errors="coerce")

# Extraer la hora, día de la semana, la semana del año, el nombre del mes y el año
usuarios_activos["hora_solicitud"] = usuarios_activos[fecha_column_activos].dt.hour.astype(str) + "_" + \
                                     usuarios_activos[fecha_column_activos].dt.strftime("%A") + "_" + \
                                     usuarios_activos[fecha_column_activos].dt.isocalendar().week.astype(str) + "_" + \
                                     usuarios_activos[fecha_column_activos].dt.strftime("%B") + "_" + \
                                     usuarios_activos[fecha_column_activos].dt.strftime("%Y")

usuarios_eliminados["hora_solicitud"] = usuarios_eliminados[fecha_column_eliminados].dt.hour.astype(str) + "_" + \
                                        usuarios_eliminados[fecha_column_eliminados].dt.strftime("%A") + "_" + \
                                        usuarios_eliminados[fecha_column_eliminados].dt.isocalendar().week.astype(str) + "_" + \
                                        usuarios_eliminados[fecha_column_eliminados].dt.strftime("%B") + "_" + \
                                        usuarios_eliminados[fecha_column_eliminados].dt.strftime("%Y")

# Verificar resultado
print(usuarios_activos[["user_id", fecha_column_activos, "hora_solicitud"]].head())
print(usuarios_eliminados[["deleted_account_id", fecha_column_eliminados, "hora_solicitud"]].head())



   user_id          created_at_cash_request               hora_solicitud
0     47.0 2019-11-19 13:57:53.511561+00:00  13_Tuesday_47_November_2019
1    804.0 2019-12-10 19:05:21.596873+00:00  19_Tuesday_50_December_2019
2    812.0 2019-12-10 19:05:48.921042+00:00  19_Tuesday_50_December_2019
3    191.0 2019-12-10 19:13:35.825460+00:00  19_Tuesday_50_December_2019
4    430.0 2019-12-10 19:14:41.668754+00:00  19_Tuesday_50_December_2019
   deleted_account_id          created_at_cash_request  \
0              1309.0 2019-12-09 14:47:35.190714+00:00   
1              4217.0 2019-12-10 22:23:12.452603+00:00   
2                91.0 2019-12-11 07:30:42.567035+00:00   
3               972.0 2019-12-11 11:40:44.978306+00:00   
4              3324.0 2019-12-11 14:01:02.506996+00:00   

                  hora_solicitud  
0     14_Monday_50_December_2019  
1    22_Tuesday_50_December_2019  
2   7_Wednesday_50_December_2019  
3  11_Wednesday_50_December_2019  
4  14_Wednesday_50_December_2019  


In [39]:
# Evitar división por cero
usuarios_activos["tasa_rechazo"] = usuarios_activos["total_operaciones_canceladas_rechazadas"] / usuarios_activos["total_solicitudes_usuario"]
usuarios_eliminados["tasa_rechazo"] = usuarios_eliminados["total_operaciones_canceladas_rechazadas"] / usuarios_eliminados["total_solicitudes_usuario"]

# Reemplazar NaN e infinitos por 0
usuarios_activos["tasa_rechazo"].replace([float("inf"), float("nan")], 0, inplace=True)
usuarios_eliminados["tasa_rechazo"].replace([float("inf"), float("nan")], 0, inplace=True)

# Verificar resultado
print(usuarios_activos[["user_id", "tasa_rechazo"]].head())
print(usuarios_eliminados[["deleted_account_id", "tasa_rechazo"]].head())


   user_id  tasa_rechazo
0     47.0      0.076923
1    804.0      1.000000
2    812.0      0.750000
3    191.0      0.500000
4    430.0      0.000000
   deleted_account_id  tasa_rechazo
0              1309.0           0.0
1              4217.0           0.0
2                91.0           1.0
3               972.0           0.5
4              3324.0           1.0


<ipython-input-39-702cbbddd062>:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  usuarios_activos["tasa_rechazo"].replace([float("inf"), float("nan")], 0, inplace=True)
<ipython-input-39-702cbbddd062>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col]

In [40]:
import pandas as pd

# Seleccionar la columna correcta de fecha
fecha_column_activos = "created_at_cash_request"
fecha_column_eliminados = "created_at_cash_request"

# Convertir a zona horaria UTC
usuarios_activos[fecha_column_activos] = pd.to_datetime(usuarios_activos[fecha_column_activos], utc=True)
usuarios_eliminados[fecha_column_eliminados] = pd.to_datetime(usuarios_eliminados[fecha_column_eliminados], utc=True)

# Calcular diferencia de días entre solicitudes y sacar el promedio por usuario
intervalo_activos = usuarios_activos.sort_values(["user_id", fecha_column_activos])\
                                    .groupby("user_id")[fecha_column_activos]\
                                    .diff().dt.days

intervalo_eliminados = usuarios_eliminados.sort_values(["deleted_account_id", fecha_column_eliminados])\
                                          .groupby("deleted_account_id")[fecha_column_eliminados]\
                                          .diff().dt.days

usuarios_activos["intervalo_promedio_solicitudes"] = usuarios_activos["user_id"]\
    .map(intervalo_activos.groupby(usuarios_activos["user_id"]).mean())

usuarios_eliminados["intervalo_promedio_solicitudes"] = usuarios_eliminados["deleted_account_id"]\
    .map(intervalo_eliminados.groupby(usuarios_eliminados["deleted_account_id"]).mean())

# Rellenar NaN con un valor alto (por ejemplo, 9999 para usuarios con una sola solicitud)
usuarios_activos["intervalo_promedio_solicitudes"].fillna(9999, inplace=True)
usuarios_eliminados["intervalo_promedio_solicitudes"].fillna(9999, inplace=True)

# Verificar resultado
print(usuarios_activos[["user_id", "intervalo_promedio_solicitudes"]].head())
print(usuarios_eliminados[["deleted_account_id", "intervalo_promedio_solicitudes"]].head())



   user_id  intervalo_promedio_solicitudes
0     47.0                       28.666667
1    804.0                     9999.000000
2    812.0                       11.000000
3    191.0                       61.000000
4    430.0                       43.285714
   deleted_account_id  intervalo_promedio_solicitudes
0              1309.0                           140.0
1              4217.0                            36.2
2                91.0                          9999.0
3               972.0                            25.0
4              3324.0                            48.0


<ipython-input-40-64d3cf623fcc>:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  usuarios_activos["intervalo_promedio_solicitudes"].fillna(9999, inplace=True)
<ipython-input-40-64d3cf623fcc>:28: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(

In [41]:
import pandas as pd

# Seleccionar las columnas correctas de fecha
fecha_column_activos = "created_at_cash_request"
fecha_column_eliminados = "created_at_cash_request"
pago_column_activos = "paid_at"
pago_column_eliminados = "paid_at"

# Asegurar que las fechas sean tipo datetime
usuarios_activos[fecha_column_activos] = pd.to_datetime(usuarios_activos[fecha_column_activos], utc=True, errors="coerce")
usuarios_activos[pago_column_activos] = pd.to_datetime(usuarios_activos[pago_column_activos], utc=True, errors="coerce")

usuarios_eliminados[fecha_column_eliminados] = pd.to_datetime(usuarios_eliminados[fecha_column_eliminados], utc=True, errors="coerce")
usuarios_eliminados[pago_column_eliminados] = pd.to_datetime(usuarios_eliminados[pago_column_eliminados], utc=True, errors="coerce")

# Calcular la diferencia en días entre pago y creación
usuarios_activos["dias_para_pago"] = (usuarios_activos[pago_column_activos] - usuarios_activos[fecha_column_activos]).dt.days
usuarios_eliminados["dias_para_pago"] = (usuarios_eliminados[pago_column_eliminados] - usuarios_eliminados[fecha_column_eliminados]).dt.days

# Definir pagos tardíos (más de 30 días)
usuarios_activos["pago_tardio"] = (usuarios_activos["dias_para_pago"] > 30).astype(int)
usuarios_eliminados["pago_tardio"] = (usuarios_eliminados["dias_para_pago"] > 30).astype(int)

# Calcular la proporción de pagos tardíos por usuario
pago_tardio_ratio_activos = usuarios_activos.groupby("user_id")["pago_tardio"].mean().reset_index()
pago_tardio_ratio_eliminados = usuarios_eliminados.groupby("deleted_account_id")["pago_tardio"].mean().reset_index()

# Renombrar columna
pago_tardio_ratio_activos.columns = ["user_id", "pago_tardio_ratio"]
pago_tardio_ratio_eliminados.columns = ["deleted_account_id", "pago_tardio_ratio"]

# Unir al dataset original
usuarios_activos = usuarios_activos.merge(pago_tardio_ratio_activos, on="user_id", how="left")
usuarios_eliminados = usuarios_eliminados.merge(pago_tardio_ratio_eliminados, on="deleted_account_id", how="left")

# Verificar resultado
print(usuarios_activos[["user_id", "pago_tardio_ratio"]].head())
print(usuarios_eliminados[["deleted_account_id", "pago_tardio_ratio"]].head())



   user_id  pago_tardio_ratio
0     47.0           0.076923
1    804.0           0.000000
2    812.0           0.000000
3    191.0           0.000000
4    430.0           0.000000
   deleted_account_id  pago_tardio_ratio
0              1309.0                0.0
1              4217.0                0.0
2                91.0                0.0
3               972.0                0.0
4              3324.0                0.0


In [42]:
import pandas as pd

# Seleccionar la columna correcta para actualizaciones
updated_column_activos = "updated_at_cash_request"
updated_column_eliminados = "updated_at_cash_request"

# Contar cuántas veces se ha modificado una solicitud
usuarios_activos["solicitudes_modificadas"] = usuarios_activos.groupby("user_id")[updated_column_activos].transform("count") - 1
usuarios_eliminados["solicitudes_modificadas"] = usuarios_eliminados.groupby("deleted_account_id")[updated_column_eliminados].transform("count") - 1

# Evitar valores negativos (en caso de usuarios con una sola solicitud)
usuarios_activos["solicitudes_modificadas"] = usuarios_activos["solicitudes_modificadas"].clip(lower=0)
usuarios_eliminados["solicitudes_modificadas"] = usuarios_eliminados["solicitudes_modificadas"].clip(lower=0)

# Verificar resultado
print(usuarios_activos[["user_id", "solicitudes_modificadas"]].head())
print(usuarios_eliminados[["deleted_account_id", "solicitudes_modificadas"]].head())



   user_id  solicitudes_modificadas
0     47.0                       12
1    804.0                        0
2    812.0                        3
3    191.0                        1
4    430.0                        7
   deleted_account_id  solicitudes_modificadas
0              1309.0                        1
1              4217.0                        5
2                91.0                        0
3               972.0                        1
4              3324.0                        2


In [47]:
# Guardar los archivos actualizados (opcional)
usuarios_activos.to_csv("drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_activos_actualizado.csv", index=False)
usuarios_eliminados.to_csv("drive/MyDrive/ColabNotebooks/Business_Payments/usuarios_eliminados_actualizado.csv", index=False)

In [43]:
usuarios_activos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29522 entries, 0 to 29521
Data columns (total 40 columns):
 #   Column                                   Non-Null Count  Dtype              
---  ------                                   --------------  -----              
 0   id_fees                                  20151 non-null  float64            
 1   cash_request_id                          20151 non-null  float64            
 2   type                                     20151 non-null  object             
 3   status_fees                              20151 non-null  object             
 4   category                                 2030 non-null   object             
 5   total_amount                             20151 non-null  float64            
 6   reason                                   20151 non-null  object             
 7   created_at_fees                          20151 non-null  object             
 8   updated_at_fees                          20151 non-null  object   

In [44]:
usuarios_eliminados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2573 entries, 0 to 2572
Data columns (total 40 columns):
 #   Column                                   Non-Null Count  Dtype              
---  ------                                   --------------  -----              
 0   id_fees                                  906 non-null    float64            
 1   cash_request_id                          906 non-null    float64            
 2   type                                     906 non-null    object             
 3   status_fees                              906 non-null    object             
 4   category                                 166 non-null    object             
 5   total_amount                             906 non-null    float64            
 6   reason                                   906 non-null    object             
 7   created_at_fees                          906 non-null    object             
 8   updated_at_fees                          906 non-null    object     

In [46]:
usuarios_eliminados.head()

,id_fees,cash_request_id,type,status_fees,category,total_amount,reason,created_at_fees,updated_at_fees,paid_at,...,mes_solicitud,semana_solicitud,dia_semana_solicitud,hora_solicitud,tasa_rechazo,intervalo_promedio_solicitudes,dias_para_pago,pago_tardio,pago_tardio_ratio,solicitudes_modificadas
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,2019-12,50_December_2019,Monday_50_December_2019,14_Monday_50_December_2019,0.0,140.0,NaN,0,0.0,1
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,2019-12,50_December_2019,Tuesday_50_December_2019,22_Tuesday_50_December_2019,0.0,36.2,NaN,0,0.0,5
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,2019-12,50_December_2019,Wednesday_50_December_2019,7_Wednesday_50_December_2019,1.0,9999.0,NaN,0,0.0,0
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,2019-12,50_December_2019,Wednesday_50_December_2019,11_Wednesday_50_December_2019,0.5,25.0,NaN,0,0.0,1
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,...,2019-12,50_December_2019,Wednesday_50_December_2019,14_Wednesday_50_December_2019,1.0,48.0,NaN,0,0.0,2
